# Pong — Actor lambda-sweep + on-trajectory preview
The Pong goal-conditioned critics (Stage-1D/1G/1L) have only ever been deployed as a per-step **argmax over the critic** (`eval/critic_local_control.critic_control`: `a_t = argmax_a f(s_t,a,g)`); Stage-1N found that controls **worse than NOOP**. The original Contrastive RL (Eysenbach et al. 2022) instead deploys a **learned actor** pi(a|s,g) maximising the critic. This trains it faithfully (`train/actor.train_actor`: `(1-lam)*E_pi[f] + lam*log pi(a_teacher) + ent_coef*H(pi)`, critic frozen) and sweeps `lam in {0 pure-critic, 0.5 critic+BC, 1 pure-BC}`.

**Runs locally on CPU** (state-based critic, fp32 — no GPU/Colab needed). It rebuilds the critic's own training tuples (state, a_teacher, goal) via the UNCHANGED pipeline (Stage-1L `reconstruct_episodes` -> Stage-1G `build_mh`), so it needs the **Stage-1J trajectory cache** `artifacts/pong_action_gate/stage1j/traj_eps0.1_<hash>.npz` and a **critic checkpoint** present (both already local). For Colab, upload those two to the repo paths first.

Env-free preview; the closed-loop verdict = `critic_local_control` with the actor swapped in for the argmax policy (separate env step).

In [ ]:
# (Colab only) clone repo; locally just run from the repo root and skip this cell.
import os
if not os.path.isdir('pong_action_gate'):
    TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
    import subprocess
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git','repo'], check=True)
    %cd repo
!git log --oneline -1
# NOTE (Colab): upload artifacts/pong_action_gate/stage1j/traj_eps0.1_e128c0ebecbafee6.npz and
#               artifacts/pong_action_gate/stage1l/critic_E01_full_seed4101.pt to those paths.

In [ ]:
# Sanity: the cache + critic must be present (the sweep reconstructs the critic's train tuples).
import os
for p in ['artifacts/pong_action_gate/stage1j/traj_eps0.1_e128c0ebecbafee6.npz',
          'artifacts/pong_action_gate/stage1l/critic_E01_full_seed4101.pt']:
    print(('OK  ' if os.path.exists(p) else 'MISSING ') + p)

In [ ]:
# FULL sweep: lam in {0, 0.5, 1} x 4000 steps on the E01_full (support-improved) critic.
# Watch BC-acc (chance 1/6=0.167) and H (entropy, ln6=1.79 uniform; should NOT crash to 0).
!PYTHONPATH=. python -u -m pong_action_gate.train.stage1o_actor_lambda_sweep \
    --critic-ckpt artifacts/pong_action_gate/stage1l/critic_E01_full_seed4101.pt \
    --epsilon 0.1 --seed 4101 --goal-dim 9 --lams 0.0,0.5,1.0 --steps 4000 --ent-coef 0.01

In [ ]:
# Read the preview. chance top1 = 1/6 = 0.167; ln6 = 1.79 uniform entropy.
#   lam=0 (pure-critic): top1_vsCrit high + val_actor near val_argmax => actor faithfully
#     reproduces the argmax-critic policy (so it inherits its closed-loop quality).
#   teacher agreement low for all lam => critic-preferred action != teacher action.
#   lam=1 (pure BC) = critic-free control baseline; closed-loop comparison is the env step.
import json
r = json.load(open('pong_action_gate/results/stage1o_actor_lambda_sweep/on_trajectory_preview.json'))
print(f"critic eps={r['epsilon']} seed={r['seed']} steps={r['steps']} tuples={r['n_train_tuples']}\n")
hdr = ['lam','top1_teach','top1_vsCrit','entropy','collapsed','val_actor','val_argmax']
print('{:>5} {:>11} {:>12} {:>8} {:>10} {:>10} {:>10}'.format(*hdr))
for lam, d in r['lambda_sweep'].items():
    print('{:>5} {:>11.3f} {:>12.3f} {:>8.3f} {:>10} {:>10.3f} {:>10.3f}'.format(
        lam, d['top1_agree_teacher'], d['top1_agree_argmax_critic'], d['actor_entropy_nats'],
        str(d['entropy_collapsed']), d['critic_value_actor_Epi_f'], d['critic_value_argmax_f']))

## Next — closed-loop verdict (env)
Swap each trained `actors/actor_lam*.pt` into `eval/critic_local_control` in place of `argmax_action`/`critic_control` (per-step `a = actor(state, goal).argmax()`), and compare agent-paddle-y / ball error vs the argmax-critic policy, NOOP, oracle, and teacher on the same H=8 anchors. That answers whether the actor parameterisation beats the greedy argmax for Pong (the B/C/D question).